# Train с новыми признаками

100 строк `train` после обеих обработок:

1. `src/preprocess.py` — удалены `search_category` и `search_is_delivery_search`,
   цена и координаты приведены к `float64` (файл `dataset/processed/train.parquet`);
2. `src/infm_params.py` — добавлены фильтры запроса `filter_*`, поля
   объявления `item_*` и текст для ретрива `item_params_text`.

Второй шаг пока не встроен в `preprocess.py`, поэтому здесь он применяется
к загруженным строкам прямо в ноутбуке.

In [0]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT / "src"))
import infm_params as ip

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)

TRAIN = ROOT / "dataset" / "processed" / "train.parquet"

## 1. Загрузка 100 строк

Читается только первый батч файла, а не все ~470 МБ.

In [1]:
train = next(pq.ParquetFile(TRAIN).iter_batches(batch_size=100)).to_pandas()

print(train.shape)
train.head()

(100, 17)


,search_query,search_location_id,search_infm_params_text,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,,Скупка б/у техники,91.0,4.989011,1.0,2303374,39.712818,652430,54.629230,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Ти...",e8b685dffe1a408e,Скупаю практически любую современную новую и б/у технику...,114
1,автоподбор,640860,Рейтинг пользователя 4 звезды и выше,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,44.051010,640860,56.273390,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советск...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем ...,114
2,баня на дровах,653240,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги К...","Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,30.128784,653240,59.783688,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтоб...",114
3,изготовление госномера на авто,634670,"Вид услуги Оборудование, производство","Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.0,2303428,40.537841,633570,45.424875,False,False,"Вид услуги Оборудование, производство Тип услуги Произво...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу на официал...,114
4,укладка плитки,658430,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ре...,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.0,44725,69.496445,658430,56.105984,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


## 2. Добавляем признаки из `infm_params_text`

In [2]:
train = ip.process_items(ip.process_queries(train))

FILTER_COLS = ["filter_category", "filter_subcategory", "filter_subject",
               "filter_online_booking"]
ITEM_COLS = ["item_category", "item_subcategory", "item_subjects",
             "item_online_booking", "item_params_text"]

print(train.shape)
print("новые колонки:", FILTER_COLS + ITEM_COLS)
train.head()

(100, 26)
новые колонки: ['filter_category', 'filter_subcategory', 'filter_subject', 'filter_online_booking', 'item_category', 'item_subcategory', 'item_subjects', 'item_online_booking', 'item_params_text']


,search_query,search_location_id,search_infm_params_text,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id,filter_category,filter_subcategory,filter_subject,filter_online_booking,item_category,item_subcategory,item_subjects,item_online_booking,item_params_text
0,скупка телевизоров,652430,,Скупка б/у техники,91.0,4.989011,1.0,2303374,39.712818,652430,54.629230,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Ти...",e8b685dffe1a408e,Скупаю практически любую современную новую и б/у технику...,114,NaN,[],[],False,No category,NaN,[],False,
1,автоподбор,640860,Рейтинг пользователя 4 звезды и выше,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,44.051010,640860,56.273390,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советск...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем ...,114,NaN,[],[],False,No category,NaN,[],False,
2,баня на дровах,653240,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги К...","Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,30.128784,653240,59.783688,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтоб...",114,"Красота, здоровье","[СПА-услуги, массаж]",[],True,"Красота, здоровье","СПА-услуги, массаж",[],True,"Красота, здоровье; СПА-услуги, массаж; Спа-процедуры; Ар..."
3,изготовление госномера на авто,634670,"Вид услуги Оборудование, производство","Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.0,2303428,40.537841,633570,45.424875,False,False,"Вид услуги Оборудование, производство Тип услуги Произво...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу на официал...,114,"Оборудование, производство",[],[],False,"Оборудование, производство","Производство, обработка",[],False,"Оборудование, производство; Производство, обработка"
4,укладка плитки,658430,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ре...,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.0,44725,69.496445,658430,56.105984,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114,Ремонт и отделка,[Ремонт квартир и домов под ключ],[],False,Ремонт и отделка,Ремонт квартир и домов под ключ,[],False,Ремонт и отделка; Ремонт квартир и домов под ключ; Все в...


## 3. Фильтры запроса: исходный текст и результат разбора

In [3]:
with pd.option_context("display.max_colwidth", 90):
    display(train[["search_query", "search_infm_params_text"] + FILTER_COLS].head(15))

,search_query,search_infm_params_text,filter_category,filter_subcategory,filter_subject,filter_online_booking
0,скупка телевизоров,,NaN,[],[],False
1,автоподбор,Рейтинг пользователя 4 звезды и выше,NaN,[],[],False
2,баня на дровах,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье","Красота, здоровье","[СПА-услуги, массаж]",[],True
3,изготовление госномера на авто,"Вид услуги Оборудование, производство","Оборудование, производство",[],[],False
4,укладка плитки,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,Ремонт и отделка,[Ремонт квартир и домов под ключ],[],False
5,сборка мебели,Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отделка,Ремонт и отделка,[Сборка и ремонт мебели],[],False
6,наращивание ногтей,"Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье","Красота, здоровье","[Маникюр, педикюр]",[],True
7,установка входных дверей,,NaN,[],[],False
8,выкуп компьютеров,Вид услуги,No category,[],[],False
9,массаж,Где вы оказываете услуги У себя дома Ваши клиенты Мужчины Кто оказывает услуги Женщина...,"Красота, здоровье","[СПА-услуги, массаж]",[],False


## 4. Поля объявления: исходный текст и результат разбора

In [4]:
with pd.option_context("display.max_colwidth", 70):
    display(train[["item_title_raw", "item_infm_params_text"] + ITEM_COLS].head(10))

,item_title_raw,item_infm_params_text,item_category,item_subcategory,item_subjects,item_online_booking,item_params_text
0,Скупка б/у техники,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимост...",No category,NaN,[],False,
1,Автоподбор Разовый осмотр автомобиля,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, ...",No category,NaN,[],False,
2,"Баня на дровах ""Прованс"" на Цветочной","Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург,...","Красота, здоровье","СПА-услуги, массаж",[],True,"Красота, здоровье; СПА-услуги, массаж; Спа-процедуры; Аренда бани;..."
3,"Изготовление дубликатов авто номеров, гос номеров","Вид услуги Оборудование, производство Тип услуги Производство, обр...","Оборудование, производство","Производство, обработка",[],False,"Оборудование, производство; Производство, обработка"
4,Ремонт и отделка квартир под ключ,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ...,Ремонт и отделка,Ремонт квартир и домов под ключ,[],False,Ремонт и отделка; Ремонт квартир и домов под ключ; Все виды отделк...
5,Сборка мебели,Вид услуги Ремонт и отделка Тип услуги Сборка и ремонт мебели Мест...,Ремонт и отделка,Сборка и ремонт мебели,[],False,Ремонт и отделка; Сборка и ремонт мебели; Ремонт шкафов; Реставрац...
6,Мастер маникюр и педикюра,"Вид услуги Красота, здоровье Место оказания услуг Москва, Новокоси...","Красота, здоровье","Маникюр, педикюр",[],True,"Красота, здоровье; Маникюр, педикюр; Аппаратный маникюр; Комбиниро..."
7,Установка межкомнатных дверей,Вид услуги Ремонт и отделка Тип услуги Двери Место оказания услуг ...,Ремонт и отделка,Двери,[],False,Ремонт и отделка; Двери; Установка межкомнатных дверей; Установка ...
8,"Выкуп пк, ноутбуков,приставок","Вид услуги Место оказания услуг ул. Братьев Кашириных, 131А Тип ст...",No category,NaN,[],False,
9,Массаж,"Вид услуги Красота, здоровье Место оказания услуг ул. Лейтенанта Ш...","Красота, здоровье","СПА-услуги, массаж",[],False,"Красота, здоровье; СПА-услуги, массаж; Массаж; Классический массаж..."


## 5. Одна строка целиком

Транспонированный вид удобнее для чтения длинных текстов.

In [5]:
with pd.option_context("display.max_colwidth", 300):
    display(train.iloc[[1]].T.rename(columns={1: "значение"}))

,значение
search_query,автоподбор
search_location_id,640860
search_infm_params_text,Рейтинг пользователя 4 звезды и выше
item_title_raw,Автоподбор Разовый осмотр автомобиля
item_rating_reviews_count,462.0
item_rating,4.982684
item_price,3500.0
item_microcat_id,2303374
item_longitude,44.05101
item_location_id,640860


## 6. Выполняет ли объявление фильтры запроса

`match_filters`: `True` — прошёл, `False` — не прошёл, `None` — фильтр не
задан.

In [6]:
checks = pd.DataFrame(
    [ip.match_filters(f, i) for f, i in zip(train[FILTER_COLS].to_dict("records"),
                                            train[ITEM_COLS].to_dict("records"))],
    index=train.index,
)
display(pd.concat([train[["search_query", "filter_category", "item_category"]], checks],
                  axis=1).head(15))

pd.DataFrame({
    "фильтр задан, строк": checks.notna().sum(),
    "объявление прошло": checks.apply(lambda s: s.dropna().astype(bool).mean()),
}).style.format({"объявление прошло": "{:.0%}"}, na_rep="—")

,search_query,filter_category,item_category,category,subcategory,subject,online_booking
0,скупка телевизоров,NaN,No category,None,None,None,None
1,автоподбор,NaN,No category,None,None,None,None
2,баня на дровах,"Красота, здоровье","Красота, здоровье",True,True,None,True
3,изготовление госномера на авто,"Оборудование, производство","Оборудование, производство",True,None,None,None
4,укладка плитки,Ремонт и отделка,Ремонт и отделка,True,True,None,None
5,сборка мебели,Ремонт и отделка,Ремонт и отделка,True,True,None,None
6,наращивание ногтей,"Красота, здоровье","Красота, здоровье",True,True,None,True
7,установка входных дверей,NaN,Ремонт и отделка,None,None,None,None
8,выкуп компьютеров,No category,No category,True,None,None,None
9,массаж,"Красота, здоровье","Красота, здоровье",True,True,None,None


,"фильтр задан, строк",объявление прошло
category,65,100%
subcategory,36,100%
subject,1,100%
online_booking,4,100%


## 7. Типы и заполненность колонок

Заполнено — не `None`/`NaN`, не пустая строка и не пустой список. Метки
«No category» / «No subcategory» и `False` считаются заполненными значениями.

In [7]:
def filled(series):
    def has_value(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return len(x) > 0
        return pd.notna(x) and x != ""
    return series.map(has_value).mean()


pd.DataFrame({
    "тип": train.dtypes.astype(str),
    "заполнено": [filled(train[c]) for c in train.columns],
}).style.format({"заполнено": "{:.0%}"})

,тип,заполнено
search_query,str,100%
search_location_id,int64,100%
search_infm_params_text,str,66%
item_title_raw,str,100%
item_rating_reviews_count,float64,99%
item_rating,float64,98%
item_price,float64,100%
item_microcat_id,int64,100%
item_longitude,float64,100%
item_location_id,int64,100%
